# 01 - Exploratory Data Analysis

This notebook performs EDA on the NASA Li-ion battery aging data and generates all required diagnostic figures for battery degradation trends.

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
METRICS_PATH = PROJECT_ROOT / 'results' / 'metrics.csv'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## Load and preprocess data

In [2]:
from src.data_loader import DEFAULT_BATTERY_IDS, download_battery_mat_files
from src.preprocessing import run_preprocessing_pipeline
from src.features import engineer_features_from_detailed

_ = download_battery_mat_files(RAW_DIR, DEFAULT_BATTERY_IDS)
all_df, detailed_df = run_preprocessing_pipeline(RAW_DIR, PROCESSED_DIR, DEFAULT_BATTERY_IDS)
feature_df = engineer_features_from_detailed(detailed_df)
feature_df.to_csv(PROCESSED_DIR / 'features_all.csv', index=False)

print('all_batteries shape:', all_df.shape)
print('features shape:', feature_df.shape)
all_df.head()

all_batteries shape: (636, 10)
features shape: (636, 24)


,battery_id,cycle_number,capacity_Ah,SOH,RUL,avg_voltage,avg_current,avg_temperature,charge_time,discharge_time
0,B0005,1,1.862203,93.110144,124,3.529829,1.818712,32.572328,7597.875,3690.234
1,B0005,2,1.852078,92.603889,123,3.537320,1.817644,32.725235,10516.000,3672.344
2,B0005,3,1.841049,92.052437,122,3.543737,1.816542,32.642862,10484.547,3651.641
3,B0005,4,1.840912,92.045600,121,3.543666,1.825618,32.514876,10397.890,3631.563
4,B0005,5,1.840360,92.017996,120,3.542343,1.826148,32.382349,10495.203,3629.172


## Capacity, SOH, and RUL trends

In [3]:
from src.visualisation import plot_capacity_fade, plot_soh_degradation, plot_rul_ground_truth

plot_capacity_fade(all_df, FIG_DIR / '01_capacity_fade_curves.png', eol_capacity=1.4)
plot_soh_degradation(all_df, FIG_DIR / '02_soh_degradation.png')
plot_rul_ground_truth(all_df, FIG_DIR / '03_rul_ground_truth.png')
print('Saved capacity/SOH/RUL plots.')

Saved capacity/SOH/RUL plots.


## Voltage profile comparison

In [4]:
from src.visualisation import plot_voltage_profile_comparison

plot_voltage_profile_comparison(
    detailed_df,
    battery_id='B0005',
    output_path=FIG_DIR / '04_voltage_profile_comparison_B0005.png',
)
print('Saved voltage profile comparison plot.')

Saved voltage profile comparison plot.


## Distribution and correlation analysis

In [5]:
from src.visualisation import (
    plot_temperature_distribution,
    plot_correlation_heatmap,
    plot_pairplot,
)

plot_temperature_distribution(all_df, FIG_DIR / '05_temperature_distribution.png')
plot_correlation_heatmap(feature_df, FIG_DIR / '06_feature_correlation_heatmap.png')
plot_pairplot(all_df, FIG_DIR / '07_key_feature_pairplot.png')
print('Saved temperature, correlation heatmap, and pairplot.')

Saved temperature, correlation heatmap, and pairplot.
